# 03 Hämta från ett API

I förra notebooken inspekterade vi Webperfs äldre lista över kommuner och regioner. Den saknar betyg. Webperf har också nyare endpoints under `/v1/` där betygen finns med och där tal är riktiga tal i stället för text.

I den här notebooken hämtar vi kommuner och regioner från de nyare endpoints, lägger dem i en tabell och kontrollerar att de stämmer med den äldre listan. Vi lär oss också att hämta data som kommer i flera sidor.

Datan är öppen under licensen CC BY-SA 4.0 med Webperf som källa.

In [ ]:
import json
import time
from pathlib import Path

import httpx
import pandas as pd

## En funktion för att hämta

Vi kommer att hämta från flera adresser på samma sätt. Därför skriver vi en egen funktion, så att vi inte behöver upprepa samma rader. En funktion definieras med `def` och kan sedan anropas med olika värden.

Funktionen gör tre saker.

1. Hämtar adressen och skickar med en header `User-Agent` som berättar vem som frågar. Det är god sed mot den som driver ett API. Headers får bara innehålla enkla tecken utan å, ä och ö, därför skriver vi for i stället för för.
2. Avbryter med ett tydligt fel om statuskoden inte är 200. Det gör `raise_for_status()`.
3. Tolkar svaret som JSON och lämnar tillbaka resultatet med `return`.

Parametern `params` är en ordbok med frågeparametrar. De hamnar efter frågetecknet i adressen.

In [ ]:
BAS = "https://api.webperf.se"
HEADERS = {"User-Agent": "webbanalys.se, kodexempel i serien Python for webbanalys"}


def hamta(sokvag, params=None):
    svar = httpx.get(BAS + sokvag, params=params, headers=HEADERS, timeout=30)
    svar.raise_for_status()
    return svar.json()

## Kommunerna

Vi hämtar alla kommuner och tittar först på nycklarna på översta nivån, precis som i förra notebooken.

In [ ]:
kommuner_svar = hamta("/v1/public/municipalities")

list(kommuner_svar.keys())

Fältet `count` är här ett heltal och inte text. Vi kontrollerar att det stämmer med antalet poster i listan.

In [ ]:
kommuner_svar["count"], len(kommuner_svar["municipalities"])

Fältet `meta` beskriver datamängden.

In [ ]:
kommuner_svar["meta"]

Lägg märke till `last_updated`. När vi körde notebooken i oktober 2026 stod det 2025-01-01, trots att betygen uppdateras löpande. Datumet verkar inte säga när datan senast ändrades. Vi litar därför inte på det, utan sparar själva datumet då vi hämtade datan.

Så här ser den första kommunen ut.

In [ ]:
kommuner_svar["municipalities"][0]

Fältnamnen skiljer sig från den äldre listan. Där hette namnet `title`, adressen `url` och länken till Wikidata `sameAs`. Här heter de `name`, `website` och `same_as`. Id-numret finns direkt som `id`.

Betygen ligger i en ordbok inuti posten, under `ratings`. Det kallas nästlad data.

## Från nästlad JSON till tabell

Funktionen `pd.json_normalize` gör om en lista med poster till en tabell. Nästlade fält blir egna kolumner, med namn som `ratings.overall`.

In [ ]:
kommuner = pd.json_normalize(kommuner_svar["municipalities"])

kommuner.head()

`head()` visar de fem första raderna. Med `dtypes` ser vi vilken datatyp varje kolumn fick. Betygen är decimaltal, `float64`, och id-numret är ett heltal, `int64`. Kolumner med text visas som `str`.

In [ ]:
kommuner.dtypes

Enligt dokumentationen kan ett betyg vara `null` om det saknas. I pandas blir det ett tomt värde. Vi räknar hur många tomma värden varje kolumn har.

In [ ]:
kommuner.isna().sum()

## Regionerna

Regionerna hämtar vi på samma sätt. Listan ligger under nyckeln `regions`.

In [ ]:
regioner_svar = hamta("/v1/public/regions")
regioner = pd.json_normalize(regioner_svar["regions"])

regioner.shape

## En gemensam tabell

Vi lägger till en kolumn som säger om raden är en kommun eller en region. Sedan staplar vi tabellerna på varandra med `pd.concat`. Med `ignore_index=True` får raderna nya löpnummer från noll.

In [ ]:
kommuner["typ"] = "kommun"
regioner["typ"] = "region"

webperf = pd.concat([kommuner, regioner], ignore_index=True)

webperf.shape

In [ ]:
webperf["typ"].value_counts()

## Data i flera sidor

Många datakällor lämnar inte ut allt på en gång. Svaret delas upp i sidor och vi får fråga efter en sida i taget. Det kallas paginering.

Webperfs sökning under `/v1/sites` fungerar så. Parametern `category` väljer kommuner med värdet 2 och regioner med värdet 3. `limit` anger hur många poster vi vill ha per sida, högst 100, och `offset` hur många poster vi hoppar över från början.

Vi hämtar första sidan med kommuner och tittar på fältet `pagination`.

In [ ]:
forsta_sidan = hamta("/v1/sites", {"category": 2, "limit": 100, "offset": 0})

forsta_sidan["pagination"]

`total` säger hur många poster som finns totalt och `has_more` om det finns fler sidor att hämta.

Nu hämtar vi alla sidor med en `while`-loop. Den fortsätter så länge villkoret är sant och avbryts med `break` när `has_more` är falskt. För varje varv ökar vi `offset` med 100. Mellan frågorna väntar vi en sekund med `time.sleep(1)` för att inte belasta servern i onödan.

In [ ]:
alla_sajter = []
offset = 0

while True:
    sida = hamta("/v1/sites", {"category": 2, "limit": 100, "offset": offset})
    alla_sajter.extend(sida["sites"])
    print("offset", offset, "gav", len(sida["sites"]), "poster")
    if not sida["pagination"]["has_more"]:
        break
    offset += 100
    time.sleep(1)

len(alla_sajter)

Vi kontrollerar att sökningen gav samma kommuner som listan vi hämtade först. Vi jämför id-numren som mängder. `True` betyder att de är exakt desamma.

In [ ]:
id_fran_sidor = {sajt["id"] for sajt in alla_sajter}

id_fran_sidor == set(kommuner["id"])

## Stämmer det med den äldre listan?

Den äldre listan från förra notebooken har inget eget fält för id-nummer. Numret finns i slutet av adressen i `uri`. Med `str.rsplit("/", n=1)` delar vi texten vid det sista snedstrecket och tar sista delen med `str[-1]`. Sedan gör vi om texten till heltal med `astype(int)`.

In [ ]:
aldre = pd.DataFrame(hamta("/public-sector/")["data"])
aldre["id"] = aldre["uri"].str.rsplit("/", n=1).str[-1].astype(int)

aldre[["id", "title", "uri"]].head()

In [ ]:
set(aldre["id"]) == set(webperf["id"])

Samma webbplatser finns alltså i båda. Kategorierna ska också stämma. Vi räknar kombinationerna av `category` i den äldre listan och vår egen kolumn `typ` med `pd.crosstab`. Varje rad i den äldre listan kopplas ihop med rätt rad i den nya via id-numret med `merge`.

In [ ]:
jamforelse = aldre.merge(webperf, on="id")

pd.crosstab(jamforelse["category"], jamforelse["typ"])

## Kontrollera en sammanställning

Webperf har också en endpoint med genomsnittsbetyg för hela offentliga sektorn. Vi jämför med medelvärdena vi själva räknar fram ur tabellen. Det är ett enkelt sätt att se att vi har hämtat hela datan och tolkat den rätt.

In [ ]:
statistik = hamta("/v1/public/stats")

statistik["statistics"]

In [ ]:
betyg = ["ratings.overall", "ratings.accessibility", "ratings.performance"]

webperf[betyg].mean().round(2)

Medelvärdena stämmer. Sammanställningen innehåller bara tre av de fem betygen, så standarder och säkerhet kan vi inte kontrollera på det här sättet.

## Spara rådata

Vi sparar svaren från de nyare endpoints som de kom, tillsammans med datumet då vi hämtade dem. Det använder vi i kommande notebooks.

`json.dumps` gör om Pythons ordböcker till JSON-text. Med `ensure_ascii=False` skrivs å, ä och ö som de är och med `indent=2` blir filen lättare att läsa.

In [ ]:
RADATA = Path("..") / "data" / "radata"
RADATA.mkdir(parents=True, exist_ok=True)

hamtad = pd.Timestamp.now(tz="Europe/Stockholm").isoformat(timespec="seconds")

for namn, innehall in [("kommuner", kommuner_svar), ("regioner", regioner_svar)]:
    fil = RADATA / f"webperf-{namn}.json"
    paket = {"hamtad": hamtad, "kalla": BAS, "svar": innehall}
    fil.write_text(json.dumps(paket, ensure_ascii=False, indent=2), encoding="utf-8")
    print(fil)

## Det här vet vi nu

- Webperfs nyare endpoints ger betygen som decimaltal och antal som heltal.
- Fältnamnen skiljer sig från den äldre listan, men webbplatserna och id-numren är desamma.
- Datumet i `meta` säger inte när datan senast ändrades, så vi sparar själva när vi hämtade den.
- Sökningen under `/v1/sites` kommer i sidor om högst 100 poster.
- Medelvärdena vi räknar fram stämmer med Webperfs egen sammanställning.

I nästa notebook använder vi länken till Wikidata för att lägga till kommunkod, län och befolkning.